# S1 — Comprendre avant de coder
## Python pour la Data Science — ESEO Pro

> **Fil rouge :** un code qui s’exécute sans erreur n’est pas une preuve que le résultat est vrai.

Cette séance suit le support **Logic Over Code**. Le notebook sert aux manipulations : environnement, NumPy, Data Detective, grain et piège de jointure.

### Objectifs
- vérifier un environnement Python reproductible ;
- manipuler des tableaux NumPy sans tunnel de syntaxe ;
- transformer une question métier en mesure vérifiable ;
- identifier le **grain** d’une table ;
- comprendre pourquoi une jointure techniquement valide peut produire un résultat métier faux ;
- utiliser l’IA comme copilote, jamais comme preuve.


## 0 — Avant Python : la question

**Challenge oral** : « Quel produit se vend le mieux ? »

Avant de coder, notez votre définition de **se vend le mieux** : quantité ? chiffre d’affaires ? marge ? nombre de commandes ?

**Règle :** mesurable ≠ pertinent.


## 1 — Build Your Lab : vérification

Le dépôt fournit `setup.sh`, `check.sh` et `run.sh`. Cette cellule vérifie seulement le kernel réellement utilisé par Jupyter.


In [ ]:
import sys, platform
from pathlib import Path
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())

ROOT = Path.cwd()
while not (ROOT / "datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "datasets").exists(), "Ouvrez ce notebook depuis le dépôt ai-course."
print("Repo root:", ROOT)


In [ ]:
import numpy as np
import pandas as pd
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("✅ Environnement prêt")


## 2 — IA locale : démonstration professeur

Prompt de démonstration Ollama :

> « Écris un programme Python qui charge `sales.csv` et calcule le chiffre d’affaires par produit. »

**Ne copiez pas aveuglément la réponse.** Demandez :
- que signifie « chiffre d’affaires » dans les colonnes disponibles ?
- y a-t-il une quantité ? des retours ? des doublons ?
- comment vérifier le total indépendamment ?

Workflow : **Generate → Run → Inspect → Question → Validate**.


## 3 — NumPy : le moteur numérique

On part d’un problème simple : prix × quantités. L’objectif n’est pas de mémoriser l’API, mais de comprendre la vectorisation.


In [ ]:
prices = np.array([12.0, 25.0, 8.0, 40.0])
quantities = np.array([3, 1, 5, 2])
revenue_by_line = prices * quantities
revenue_by_line


In [ ]:
print("shape:", revenue_by_line.shape)
print("dtype:", revenue_by_line.dtype)
print("total:", revenue_by_line.sum())
print("mean:", revenue_by_line.mean())
print("max:", revenue_by_line.max())
assert np.isclose(revenue_by_line.sum(), 181.0)
print("✅ Contrôle indépendant attendu : 181.0")


### Mini-challenge NumPy
Sans boucle `for`, calculez le prix TTC avec +20 %, puis le CA TTC par ligne. Expliquez ce que fait chaque opération.


In [ ]:
# Correction à dévoiler après tentative
prices_ttc = prices * 1.20
revenue_ttc = prices_ttc * quantities
print(prices_ttc)
print(revenue_ttc)
print("CA TTC:", revenue_ttc.sum())


## 4 — Data Detective

Mission par équipe : **« Quel produit se vend le mieux ? »**

Avant tout calcul :
1. définissez « se vend le mieux » ;
2. dites ce que représente **une ligne** de chaque fichier ;
3. identifiez les clés ;
4. choisissez une mesure ;
5. prévoyez un contrôle indépendant.


In [ ]:
DATA = ROOT / "datasets" / "retail_case"
customers = pd.read_csv(DATA / "customers.csv")
orders = pd.read_csv(DATA / "orders.csv")
order_lines = pd.read_csv(DATA / "order_lines.csv")
products = pd.read_csv(DATA / "products.csv")

for name, df in {"customers": customers, "orders": orders, "order_lines": order_lines, "products": products}.items():
    print(f"{name:12s} shape={df.shape} columns={list(df.columns)}")
    display(df.head(3))


### Stop avant de continuer
Écrivez vos réponses.

Grains attendus :
- `customers` : une ligne = un client ;
- `orders` : une ligne = une commande ;
- `order_lines` : une ligne = un produit dans une commande ;
- `products` : une ligne = un produit.

La table de faits naturelle pour le CA est au **grain ligne de commande**.


In [ ]:
# Vérifions les hypothèses de clés avant la jointure.
print("customers.customer_id unique:", customers["customer_id"].is_unique)
print("orders.order_id unique:", orders["order_id"].is_unique)
print("products.product_id unique:", products["product_id"].is_unique)
print("order_lines (order_id, line_id) unique:", not order_lines.duplicated(["order_id", "line_id"]).any())


## 5 — Construire le fait de vente

On déclare les cardinalités attendues avec `validate=` : ce n’est pas du confort, c’est une hypothèse métier rendue testable.


In [ ]:
fact_sales = (
    order_lines
    .merge(orders, on="order_id", how="left", validate="many_to_one")
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(products[["product_id", "product_name", "category"]], on="product_id", how="left", validate="many_to_one")
)
fact_sales["revenue"] = fact_sales["quantity"] * fact_sales["unit_price"]

reference_revenue = fact_sales["revenue"].sum()
print("Rows:", len(fact_sales))
print("Reference revenue:", reference_revenue)
display(fact_sales.head())


### Répondre à la question métier
Voici deux définitions légitimes et potentiellement différentes : quantité vendue et chiffre d’affaires.


In [ ]:
by_product = (fact_sales.groupby(["product_id", "product_name"], as_index=False)
              .agg(quantity_sold=("quantity", "sum"), revenue=("revenue", "sum"))
              .sort_values("revenue", ascending=False))
display(by_product)
print("Leader CA:", by_product.iloc[0]["product_name"])
print("Leader quantité:", by_product.sort_values("quantity_sold", ascending=False).iloc[0]["product_name"])


## 6 — Le piège de jointure

Un fichier marketing associe plusieurs tags à certains clients. La jointure suivante **s’exécute parfaitement**. Est-elle correcte pour calculer le CA total ?


In [ ]:
customer_tags = pd.DataFrame({
    "customer_id": ["C001", "C001", "C002", "C002", "C003", "C003", "C004", "C004", "C005", "C005"],
    "tag": ["newsletter", "vip", "newsletter", "promo", "organic", "loyalty", "partner", "vip", "organic", "promo"]
})

bad = fact_sales.merge(customer_tags, on="customer_id", how="left")
bad_revenue = bad["revenue"].sum()
print("Rows before:", len(fact_sales), "| after:", len(bad))
print("CA before:", reference_revenue, "| after:", bad_revenue)
print("Inflation:", round(bad_revenue / reference_revenue, 3), "x")
assert len(bad) > len(fact_sales)
assert bad_revenue > reference_revenue
print("⚠️ Aucun crash Python. Résultat métier faux.")


### Faire échouer explicitement une mauvaise hypothèse
Si vous pensiez que chaque client n’avait qu’un tag, dites-le à Pandas.


In [ ]:
try:
    fact_sales.merge(customer_tags, on="customer_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__ + ":", e)


## 7 — Contrat de données minimal

Quelques invariants simples peuvent empêcher une erreur silencieuse.


In [ ]:
checks = {
    "customer_id_unique": customers["customer_id"].is_unique,
    "order_id_unique": orders["order_id"].is_unique,
    "product_id_unique": products["product_id"].is_unique,
    "order_line_composite_key_unique": not order_lines.duplicated(["order_id", "line_id"]).any(),
    "quantities_positive": (order_lines["quantity"] > 0).all(),
    "prices_non_negative": (order_lines["unit_price"] >= 0).all(),
    "fact_rows_preserved": len(fact_sales) == len(order_lines),
}
for k, v in checks.items():
    print(("✅" if v else "❌"), k)
assert all(checks.values())


## 8 — Exit ticket

Répondez en trois phrases :
1. Quelle question faut-il poser **avant** de coder ?
2. Donnez un exemple de proxy mesurable mais trompeur.
3. Quel résultat obtenu aujourd’hui pourriez-vous vérifier d’une autre manière ?

---

# À retenir

**Mesurable ≠ pertinent.**  
**L’IA peut produire le code. Vous restez responsables du résultat.**  
**Si vous ne savez pas ce qu’une ligne représente, ne calculez rien.**  
**LE CODE TOURNE N’EST PAS UN TEST DE VÉRITÉ.**
